# Paraxial Optical-System Laboratory

Build an optical system from **translation**, **refraction**, **reflection**, **thin-lens**, and **thick-lens** components while keeping every matrix factor visible.

This notebook preserves the course convention

\[
\mathbf r=\begin{pmatrix}n\alpha\\x\end{pmatrix},\qquad
T=\begin{pmatrix}1&0\\d/n&1\end{pmatrix},\quad
R_a=\begin{pmatrix}1&-\mathcal P\\0&1\end{pmatrix},\quad
R_e=\begin{pmatrix}1&2n/R\\0&1\end{pmatrix}.
\]

Each thin or thick lens contributes **one** lens matrix to the product. The first component added is **rightmost** and multiplies the input ray first; later components pile on the left:

\[
M_{VV'}=M_n\cdots M_2 M_1.
\]

Because matrix multiplication is not commutative, reordering a component changes \(M_{VV'}\). Principal and conjugate planes are always calculated from that resolved base matrix.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import importlib
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import paraxial_config as config
import paraxial_dashboard_tools as visuals
import paraxial_engine as engine
import paraxial_style as style
import paraxial_tools as tools

# Frozen dataclasses (e.g. MatrixFactor.style_key) often stay stale under
# autoreload alone — force dependency order so the kernel matches disk.
config = importlib.reload(config)
engine = importlib.reload(engine)
tools = importlib.reload(tools)
visuals = importlib.reload(visuals)
style = importlib.reload(style)
ParaxialDashboard = style.ParaxialDashboard

print("Python:", sys.executable)
for module in (np, plt.matplotlib, widgets):
    print(f"  {module.__name__}: {module.__version__}")
print(
    "Paraxial matrix, principal-plane, and conjugate-plane tools loaded OK "
    f"(MatrixFactor fields: {tuple(engine.MatrixFactor.__dataclass_fields__)})."
)


Python: d:\GitHub\Optics\.venv\Scripts\python.exe
  numpy: 2.4.6
  matplotlib: 3.11.0
  ipywidgets: 8.1.8
Paraxial matrix, principal-plane, and conjugate-plane tools loaded OK (MatrixFactor fields: ('label', 'matrix', 'description', 'style_key')).


## 1. Source regression: lens matrices and product order

The following cell checks the original interface-power and translation examples, rebuilds the three thick lenses from the course radii, and verifies the MATLAB-written product

\[
M_t=M_1\,T\,M_2\,T\,M_3\approx\begin{pmatrix}0.5616&-0.0574\\11.9278&0.5622\end{pmatrix}.
\]

It then builds the interactive stack (first component rightmost) and prints that product order.

In [2]:
assert np.isclose(engine.interface_power(1.3, 1.0, 0.50), -0.6)
np.testing.assert_allclose(
    engine.translation_matrix(1.0, 1.5),
    [[1.0, 0.0], [1.5, 1.0]],
)

p1 = engine.interface_power(1.0, 1.812, 11.5)
p2 = engine.interface_power(1.812, 1.0, -127.0)
p3 = engine.interface_power(1.0, 1.695, -23.5)
p4 = engine.interface_power(1.695, 1.0, 10.2)
p5 = engine.interface_power(1.0, 1.812, 30.0)
p6 = engine.interface_power(1.812, 1.0, -15.0)

def matlab_thick(n, d, ps, pf):
    """Source MATLAB order: Ra(Ps)*T*Ra(Pf) with first factor leftmost."""
    return engine.cascade(
        (
            engine.refraction_matrix(ps),
            engine.translation_matrix(n, d),
            engine.refraction_matrix(pf),
        )
    )

m1 = matlab_thick(1.812, 5.00, p1, p2)
m2 = matlab_thick(1.695, 1.55, p3, p4)
m3 = matlab_thick(1.812, 5.00, p5, p6)
matlab_total = engine.cascade(
    (
        m1,
        engine.translation_matrix(1.0, 1.25),
        m2,
        engine.translation_matrix(1.0, 2.50),
        m3,
    )
)
expected_matlab = np.array([[0.5616, -0.0574], [11.9278, 0.5622]])
np.testing.assert_allclose(matlab_total, expected_matlab, atol=5e-5)
assert np.isclose(engine.assert_unit_determinant(matlab_total), 1.0)

# Interactive thick_lens_matrix lists optical order (Ps, T, Pf), then reverses once.
optical = (
    engine.refraction_matrix(p1),
    engine.translation_matrix(1.812, 5.00),
    engine.refraction_matrix(p2),
)
np.testing.assert_allclose(
    engine.thick_lens_matrix(1.812, 5.00, p1, p2),
    engine.cascade(reversed(optical)),
)

course_elements = tools.preset_elements("course_exercise")
course_system = tools.build_system(course_elements)
assert len(course_system.factors) == 5
assert course_system.product_expression.endswith("Mtk[L1]")
assert course_system.is_unit_determinant

print("MATLAB-written product M1 @ T @ M2 @ T @ M3 =")
print(matlab_total)
print("\nInteractive thick lens equals cascade(reversed(Ra_s, T, Ra_f)):")
print(engine.thick_lens_matrix(1.812, 5.00, p1, p2))
print("\nInteractive stack product (first component rightmost):")
print(course_system.product_expression)
for factor in course_system.factors:
    print(f"{factor.label} — {factor.description}")
    print(factor.matrix)
print("\nM_VV' =")
print(course_system.matrix)
print("det(M_VV') =", course_system.determinant)

MATLAB-written product M1 @ T @ M2 @ T @ M3 =
[[ 0.56164034 -0.05736564]
 [11.92782276  0.56219763]]

Interactive thick lens equals cascade(reversed(Ra_s, T, Ra_f)):
[[ 0.98235734 -0.07575667]
 [ 2.7593819   0.80516364]]

Interactive stack product (first component rightmost):
Mtk[L3] @ T[Gap2] @ Mtk[L2] @ T[Gap1] @ Mtk[L1]
Mtk[L3] — n=1.812, d=5 m, P₁=0.0270667 m⁻¹, P₂=0.0541333 m⁻¹
[[ 0.85062546 -0.07715693]
 [ 2.7593819   0.92531273]]
T[Gap2] — n=1, d=2.5 m
[[1.  0. ]
 [2.5 1. ]]
Mtk[L2] — n=1.695, d=1.55 m, P₁=-0.0295745 m⁻¹, P₂=-0.0681373 m⁻¹
[[1.0623084  0.09955446]
 [0.91445428 1.0270445 ]]
T[Gap1] — n=1, d=1.25 m
[[1.   0.  ]
 [1.25 1.  ]]
Mtk[L1] — n=1.812, d=5 m, P₁=0.0706087 m⁻¹, P₂=0.0063937 m⁻¹
[[ 0.98235734 -0.07575667]
 [ 2.7593819   0.80516364]]

M_VV' =
[[ 0.56219763 -0.05736564]
 [11.92782276  0.56164034]]
det(M_VV') = 1.0


## 2. Principal and conjugate plane sanity check

For a thin lens with \(\mathcal P=10\,\mathrm{m}^{-1}\), the next cell resolves

\[
T(V'\to H')\,M_{VV'}\,T(H\to V)=M_{HH'}
\]

and then uses an Object→V distance \(x=0.2\,\mathrm m\). For a thin lens
\(D=D'=0\), so \(s=x-D=0.2\,\mathrm m\). The assertions verify the
principal-plane reduction, the vertex-based conjugate path, and
\(M_{21}=0\).


In [3]:
lens_system = tools.build_system(
    [tools.OpticalElement.thin_lens("L1", 5.0, 5.0)]
)
cardinal = engine.principal_planes(lens_system.matrix)
# Thin lens: D = D' = 0, so Object→V distance x equals principal distance s.
conjugate = engine.conjugate_from_vertex_distance(
    cardinal,
    mode="object_to_v",
    distance=0.2,
)

np.testing.assert_allclose(
    cardinal.reduction_matrix,
    cardinal.equivalent_matrix,
    atol=config.MATRIX_TOLERANCE,
)
assert conjugate.is_conjugate
assert np.isclose(conjugate.matrix[1, 0], 0.0, atol=config.MATRIX_TOLERANCE)
assert np.isclose(conjugate.lagrange_invariant, 1.0)
assert np.isclose(conjugate.object_distance, 0.2)
assert np.isclose(conjugate.object_to_vertex, 0.2)

print("Principal reduction:", cardinal.principal_product_expression)
print(cardinal.reduction_matrix)
print(
    f"P={cardinal.power:.6g} m^-1, D={cardinal.object_principal_offset:.6g} m, "
    f"D'={cardinal.image_principal_offset:.6g} m"
)
print("\nConjugate product:", conjugate.product_expression)
print(conjugate.matrix)
print(
    f"x={conjugate.object_to_vertex:.6g} m, s={conjugate.object_distance:.6g} m, "
    f"s'={conjugate.image_distance:.6g} m, x'={conjugate.exit_vertex_to_image:.6g} m, "
    f"mx={conjugate.lateral_magnification:.6g}, "
    f"m_alpha={conjugate.angular_magnification:.6g}, M21={conjugate.conjugacy_residual:.3g}"
)


Principal reduction: T(V'→H') @ M_VV' @ T(H→V)
[[  1. -10.]
 [  0.   1.]]
P=10 m^-1, D=-0 m, D'=-0 m

Conjugate product: T(H'→image) @ M_HH' @ T(object→H)
[[ -1. -10.]
 [  0.  -1.]]
x=0.2 m, s=0.2 m, s'=0.2 m, x'=0.2 m, mx=-1, m_alpha=-1, M21=0


## 3. Interactive system builder

Use the controls below to:

1. load a preset or add translation, refraction, reflection, thin-lens, and thick-lens components;
2. move components with the arrow buttons and immediately see the non-commutative product change;
3. inspect each component’s single 2×2 matrix and the resolved vertex-to-vertex matrix \(M_{VV'}\) (first added = rightmost);
4. inspect the principal-plane values and the full reduction to \(M_{HH'}\);
5. choose Object→V or V′→image and inspect the conjugate product, \(M_{21}=0\), magnifications, image classification, and ray schematic.

A thin or thick lens adds exactly one lens matrix to the product.


In [4]:
dashboard = ParaxialDashboard()
dashboard.show()

In [5]:
f1=0.1
f2=0.2
p1=1/f1
p2=1/f2

print(p1,p2)

10.0 5.0
